# 📂 HarMeme OCR — Notebook 1: Splitter
Reads all annotations from **Harm-C** and **Harm-P**, collects every unique image
filename + its source image folder, then splits them into `N_CHUNKS` chunk CSVs.

**Run this once → download `/kaggle/working` → upload as Kaggle dataset `harmeme-chunks`**


In [ ]:
import os, json, math
import pandas as pd
from pathlib import Path

# ── Paths ─────────────────────────────────────────────────────────────────────
HARMC_ROOT = Path("/kaggle/input/harmeme/MINI_PROJECT_2/Harm-C/datasets/memes/defaults")
HARMP_ROOT = Path("/kaggle/input/harmeme/MINI_PROJECT_2/Harm-P/datasets/memes/defaults")
OUTPUT_DIR = Path("/kaggle/working")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

N_CHUNKS = 4   # run 4 parallel chunk-processor notebooks

# ── Verify paths ──────────────────────────────────────────────────────────────
for root, name in [(HARMC_ROOT, "Harm-C"), (HARMP_ROOT, "Harm-P")]:
    img_dir   = root / "images"
    annot_dir = root / "annotations"
    assert img_dir.exists(),   f"❌ {name} images not found: {img_dir}"
    assert annot_dir.exists(), f"❌ {name} annotations not found: {annot_dir}"
    n_imgs = len(list(img_dir.glob("*")))
    print(f"✅ {name}: {n_imgs} images | annotations: {[f.name for f in annot_dir.glob('*.jsonl')]}")


In [ ]:
# ── Collect all rows from Harm-C and Harm-P annotations ──────────────────────
all_rows = []

for root, tag in [(HARMC_ROOT, "harm-c"), (HARMP_ROOT, "harm-p")]:
    annot_dir = root / "annotations"
    img_root  = root / "images"

    for jsonl_file in sorted(annot_dir.glob("*.jsonl")):
        with open(jsonl_file) as f:
            rows = [json.loads(line) for line in f if line.strip()]

        for r in rows:
            # strip any leading path from image field e.g. "img/00123.png" → "00123.png"
            fname = str(r.get("image", "")).replace("img/", "").replace("img\\", "").split("/")[-1].split("\\")[-1]
            r["fname"]    = fname
            r["img_root"] = str(img_root)   # absolute path to image folder
            r["source"]   = tag

        all_rows.extend(rows)
        print(f"   [{tag}] {jsonl_file.name:15s}: {len(rows):5d} rows")

df = pd.DataFrame(all_rows)

print(f"\nTotal rows   : {len(df)}")
print(f"Harm-C rows  : {(df['source'] == 'harm-c').sum()}")
print(f"Harm-P rows  : {(df['source'] == 'harm-p').sum()}")
print(f"\nColumns: {list(df.columns)}")
print(df[['fname', 'img_root', 'source']].head(5))


In [ ]:
# ── Deduplicate by fname (same image may appear in train+val) ─────────────────
df_unique = df[["fname", "img_root", "source"]].drop_duplicates(subset="fname").reset_index(drop=True)

print(f"Unique images: {len(df_unique)}")
print(f"  Harm-C : {(df_unique['source'] == 'harm-c').sum()}")
print(f"  Harm-P : {(df_unique['source'] == 'harm-p').sum()}")

# Quick sanity — spot check 5 images exist on disk
missing = []
for _, row in df_unique.head(20).iterrows():
    if not (Path(row["img_root"]) / row["fname"]).exists():
        missing.append(row["fname"])
if missing:
    print(f"\n⚠️  Some images not found on disk (first 20 checked): {missing}")
else:
    print("\n✅ Spot-check passed — all sampled images found on disk.")


In [ ]:
# ── Split into N_CHUNKS and save chunk CSVs ───────────────────────────────────
chunk_size = math.ceil(len(df_unique) / N_CHUNKS)

for i in range(N_CHUNKS):
    chunk    = df_unique.iloc[i * chunk_size : (i + 1) * chunk_size].copy()
    csv_path = OUTPUT_DIR / f"harmeme_chunk_{i}.csv"
    chunk.to_csv(csv_path, index=False)
    print(f"  Chunk {i}: {len(chunk):5d} images → {csv_path}")

print(f"\n✅ {N_CHUNKS} chunk CSVs saved to {OUTPUT_DIR}")
print("\nNext steps:")
print("  1. Download /kaggle/working (contains harmeme_chunk_0.csv … harmeme_chunk_3.csv)")
print("  2. Upload as a new Kaggle dataset named  'harmeme-chunks'")
print("  3. Run Notebook 2 (chunk processor) with CHUNK_ID = 0, 1, 2, 3 in parallel")
